# Dental Anomaly Detection — Fine-Tuning Notebook

**Final Year Project Training Pipeline**

This notebook fine-tunes your existing Faster R-CNN model on `Dataset_Batch2` using Google Colab's free GPU.

## Quick Start
1. **Runtime → Change runtime type → GPU (T4)**
2. Click **Run All**
3. Upload `dental_anomaly_detector.pth` when prompted
4. Upload `Dataset_Batch2.rar` when prompted
5. Wait ~1–2 hours for training
6. `dental_model_v2.pth` downloads automatically

## Output
Place the downloaded `dental_model_v2.pth` in the **`models/`** folder on Replit.

## 1. Install Dependencies

In [ ]:
!pip install -q albumentations opencv-python-headless rarfile scikit-learn matplotlib pandas
!apt-get -qq install -y unrar-free 2>/dev/null || apt-get -qq install -y unrar 2>/dev/null

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Upload Files

Upload your pre-trained model and dataset RAR archive.

In [ ]:
from google.colab import files
import os
import shutil
import rarfile

WORK_DIR = "/content/dental_training"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print("Upload dental_anomaly_detector.pth (pre-trained model):")
uploaded_model = files.upload()
MODEL_PATH = list(uploaded_model.keys())[0]
print(f"Saved: {MODEL_PATH}")

print("\nUpload Dataset_Batch2.rar:")
uploaded_data = files.upload()
RAR_PATH = list(uploaded_data.keys())[0]
print(f"Saved: {RAR_PATH}")

# Extract RAR archive
DATASET_DIR = os.path.join(WORK_DIR, "Dataset_Batch2")
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

print("Extracting RAR file...")
with rarfile.RarFile(RAR_PATH) as rf:
    rf.extractall(WORK_DIR)

# Find extracted root (may be nested)
for root, dirs, files_in_dir in os.walk(WORK_DIR):
    if "train" in dirs and "valid" in dirs:
        DATASET_DIR = root
        break

print(f"Dataset root: {DATASET_DIR}")
for split in ["train", "valid", "test"]:
    split_dir = os.path.join(DATASET_DIR, split)
    csv_path = os.path.join(split_dir, "_annotations.csv")
    n_imgs = len([f for f in os.listdir(split_dir) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]) if os.path.isdir(split_dir) else 0
    print(f"  {split}: {n_imgs} images, CSV exists: {os.path.exists(csv_path)}")

## 3. Configuration & Label Mapping

Maps new dataset classes (0–8) to the 7 anomaly classes used by the model.
Classes 0 (Healthy) and 8 are **skipped**.

In [ ]:
import numpy as np
import pandas as pd
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from collections import defaultdict
import copy
import time
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

# --- Label mapping: new dataset class → model class ---
LABEL_MAP = {
    0: None,   # Healthy → SKIP
    1: 1,      # Caries
    2: 2,      # Impacted
    3: 4,      # Infection
    4: 5,      # Fractured
    5: 3,      # Broken Crown
    6: 6,      # Periodontal (NEW)
    7: 7,      # Other (NEW)
    8: None,   # SKIP
}

CLASS_NAMES = {
    1: "Caries",
    2: "Impacted Teeth",
    3: "Broken Down Crown/Root",
    4: "Infection",
    5: "Fractured Teeth",
    6: "Periodontal Bone Loss",
    7: "Other Abnormalities",
}

NUM_CLASSES = 8          # background (0) + 7 anomaly classes
IMAGE_SIZE = 416
BATCH_SIZE = 4
NUM_EPOCHS = 25
LR_BACKBONE = 0.00005
LR_HEAD = 0.0005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {DEVICE}")

## 4. Dataset Loading

Reads `_annotations.csv` from each split folder and converts to PyTorch detection format.

In [ ]:
def parse_class_id(raw_class):
    """Convert CSV class column to integer ID."""
    if isinstance(raw_class, str):
        raw_class = raw_class.strip()
        if raw_class.isdigit():
            return int(raw_class)
        # Try matching class name
        name_to_id = {v.lower(): k for k, v in CLASS_NAMES.items()}
        return name_to_id.get(raw_class.lower(), -1)
    return int(raw_class)


def load_annotations(csv_path, images_dir):
    """Load CSV and group annotations by filename."""
    df = pd.read_csv(csv_path)
    # Normalize column names
    df.columns = [c.strip().lower() for c in df.columns]

    grouped = defaultdict(list)
    skipped = 0

    for _, row in df.iterrows():
        filename = str(row.get("filename", row.get("image", ""))).strip()
        if not filename:
            continue

        raw_cls = row.get("class", row.get("class_id", -1))
        src_id = parse_class_id(raw_cls)
        mapped = LABEL_MAP.get(src_id)

        if mapped is None:
            skipped += 1
            continue

        xmin = float(row.get("xmin", row.get("x_min", 0)))
        ymin = float(row.get("ymin", row.get("y_min", 0)))
        xmax = float(row.get("xmax", row.get("x_max", 0)))
        ymax = float(row.get("ymax", row.get("y_max", 0)))

        if xmax <= xmin or ymax <= ymin:
            continue

        grouped[filename].append({
            "bbox": [xmin, ymin, xmax, ymax],
            "label": mapped,
        })

    # Only keep images that exist on disk AND have at least one annotation
    samples = []
    for filename, anns in grouped.items():
        img_path = os.path.join(images_dir, filename)
        if os.path.isfile(img_path) and len(anns) > 0:
            samples.append({"filename": filename, "path": img_path, "annotations": anns})

    print(f"  Loaded {len(samples)} images, skipped {skipped} annotations (Healthy/Class 8)")
    return samples


class DentalDataset(Dataset):
    """PyTorch dataset for dental X-ray object detection."""

    def __init__(self, samples, transforms=None):
        self.samples = samples
        self.transforms = transforms

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = cv2.imread(sample["path"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        bboxes = [a["bbox"] for a in sample["annotations"]]
        labels = [a["label"] for a in sample["annotations"]]

        if self.transforms:
            transformed = self.transforms(
                image=image,
                bboxes=bboxes,
                class_labels=labels,
            )
            image = transformed["image"]
            bboxes = transformed["bboxes"]
            labels = transformed["class_labels"]

        # Convert to PyTorch tensors
        if not isinstance(image, torch.Tensor):
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0

        boxes = torch.as_tensor(bboxes, dtype=torch.float32)
        labels_t = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels_t,
            "image_id": torch.tensor([idx]),
        }
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


# Load all splits
print("Loading datasets...")
train_samples = load_annotations(os.path.join(DATASET_DIR, "train", "_annotations.csv"),
                               os.path.join(DATASET_DIR, "train"))
valid_samples = load_annotations(os.path.join(DATASET_DIR, "valid", "_annotations.csv"),
                                 os.path.join(DATASET_DIR, "valid"))
test_samples  = load_annotations(os.path.join(DATASET_DIR, "test", "_annotations.csv"),
                                 os.path.join(DATASET_DIR, "test"))
print(f"Train: {len(train_samples)}, Valid: {len(valid_samples)}, Test: {len(test_samples)}")

## 5. Data Augmentation (On-the-Fly)

Applied only during training. Validation/test use resize + normalize only.

In [ ]:
def get_train_transforms():
    return A.Compose(
        [
            A.RandomRotate90(p=0.5),
            A.HorizontalFlip(p=0.5),
            A.Affine(
                scale=(0.85, 1.15),
                rotate=(-30, 30),
                p=0.7,
            ),
            A.RandomBrightnessContrast(p=0.5),
            A.GaussNoise(p=0.3),
            A.ElasticTransform(alpha=1, sigma=50, p=0.3),
            A.Resize(IMAGE_SIZE, IMAGE_SIZE),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ],
        bbox_params=A.BboxParams(
            format="pascal_voc",
            label_fields=["class_labels"],
            min_visibility=0.3,
        ),
    )


def get_val_transforms():
    return A.Compose(
        [
            A.Resize(IMAGE_SIZE, IMAGE_SIZE),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ],
        bbox_params=A.BboxParams(
            format="pascal_voc",
            label_fields=["class_labels"],
        ),
    )


train_dataset = DentalDataset(train_samples, transforms=get_train_transforms())
valid_dataset = DentalDataset(valid_samples, transforms=get_val_transforms())
test_dataset  = DentalDataset(test_samples,  transforms=get_val_transforms())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Valid: {len(valid_loader)}, Test: {len(test_loader)}")

## 6. Load Pre-trained Model

Loads `dental_anomaly_detector.pth`, replaces the classifier head for 7 classes,
and transfers all compatible weights.

In [ ]:
def build_model(num_classes=NUM_CLASSES):
    model = fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def load_pretrained_model(model_path, num_classes=NUM_CLASSES):
    model = build_model(num_classes)
    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    # Load all compatible weights; skip classifier head (different num classes)
    model_dict = model.state_dict()
    pretrained = {k: v for k, v in state_dict.items()
                  if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(pretrained)
    model.load_state_dict(model_dict)

    print(f"Loaded {len(pretrained)}/{len(model_dict)} weight tensors from pre-trained model")
    print(f"Skipped classifier head keys (replaced for {num_classes} classes)")
    return model


model = load_pretrained_model(MODEL_PATH)
model.to(DEVICE)

## 7. Training

- **Optimizer**: SGD (momentum 0.9, weight decay 0.0005)
- **Learning rates**: backbone 0.00005, ROI heads 0.0005
- **Epochs**: 25
- **Best model**: saved based on lowest validation loss
- **Empty batches**: skipped automatically

In [ ]:
def get_optimizer(model):
    """Different learning rates for backbone vs ROI heads."""
    params_backbone = []
    params_head = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "backbone" in name:
            params_backbone.append(param)
        else:
            params_head.append(param)
    return torch.optim.SGD(
        [
            {"params": params_backbone, "lr": LR_BACKBONE},
            {"params": params_head, "lr": LR_HEAD},
        ],
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )


def train_one_epoch(model, loader, optimizer, epoch):
    model.train()
    total_loss = 0.0
    num_batches = 0
    skipped = 0

    for images, targets in loader:
        # Skip batches with empty targets
        valid = [(img, tgt) for img, tgt in zip(images, targets)
                 if len(tgt["boxes"]) > 0]
        if not valid:
            skipped += 1
            continue

        images = [img.to(DEVICE) for img, _ in valid]
        targets = [{k: v.to(DEVICE) for k, v in tgt.items()} for _, tgt in valid]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
        num_batches += 1

    avg_loss = total_loss / max(num_batches, 1)
    print(f"  Epoch {epoch} Train Loss: {avg_loss:.4f} (skipped {skipped} empty batches)")
    return avg_loss


@torch.no_grad()
def evaluate_loss(model, loader):
    model.train()  # Faster R-CNN returns losses only in train mode
    total_loss = 0.0
    num_batches = 0

    for images, targets in loader:
        valid = [(img, tgt) for img, tgt in zip(images, targets)
                 if len(tgt["boxes"]) > 0]
        if not valid:
            continue

        images = [img.to(DEVICE) for img, _ in valid]
        targets = [{k: v.to(DEVICE) for k, v in tgt.items()} for _, tgt in valid]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()
        num_batches += 1

    return total_loss / max(num_batches, 1)


optimizer = get_optimizer(model)
train_losses = []
val_losses = []
best_val_loss = float("inf")
best_model_state = None

print("Starting training...")
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    t_loss = train_one_epoch(model, train_loader, optimizer, epoch)
    v_loss = evaluate_loss(model, valid_loader)

    train_losses.append(t_loss)
    val_losses.append(v_loss)

    print(f"  Epoch {epoch} Val Loss: {v_loss:.4f}")

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_model_state = copy.deepcopy(model.state_dict())
        print(f"  ★ New best model (val loss: {best_val_loss:.4f})")

elapsed = time.time() - start_time
print(f"\nTraining complete in {elapsed/60:.1f} minutes. Best val loss: {best_val_loss:.4f}")

## 8. Training Metrics — Loss Curves

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
epochs = range(1, len(train_losses) + 1)
ax.plot(epochs, train_losses, "b-o", label="Train Loss", markersize=4)
ax.plot(epochs, val_losses, "r-o", label="Val Loss", markersize=4)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Test Set Evaluation — Precision, Recall, F1

In [ ]:
def box_iou(box1, box2):
    """Compute IoU between two boxes [x1,y1,x2,y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0


@torch.no_grad()
def evaluate_test(model, loader, iou_threshold=0.5, conf_threshold=0.5):
    model.eval()
    per_class_tp = defaultdict(int)
    per_class_fp = defaultdict(int)
    per_class_fn = defaultdict(int)

    for images, targets in loader:
        images_gpu = [img.to(DEVICE) for img in images]
        outputs = model(images_gpu)

        for output, target in zip(outputs, targets):
            gt_boxes = target["boxes"].numpy()
            gt_labels = target["labels"].numpy()

            mask = output["scores"].cpu().numpy() >= conf_threshold
            pred_boxes = output["boxes"].cpu().numpy()[mask]
            pred_labels = output["labels"].cpu().numpy()[mask]
            pred_scores = output["scores"].cpu().numpy()[mask]

            matched_gt = set()

            for pb, pl, ps in sorted(zip(pred_boxes, pred_labels, pred_scores),
                                     key=lambda x: -x[2]):
                best_iou, best_idx = 0, -1
                for gi, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
                    if gi in matched_gt or gl != pl:
                        continue
                    iou = box_iou(pb, gb)
                    if iou > best_iou:
                        best_iou, best_idx = iou, gi

                if best_iou >= iou_threshold:
                    per_class_tp[int(pl)] += 1
                    matched_gt.add(best_idx)
                else:
                    per_class_fp[int(pl)] += 1

            for gi, gl in enumerate(gt_labels):
                if gi not in matched_gt:
                    per_class_fn[int(gl)] += 1

    # Compute metrics per class
    results = {}
    all_classes = set(list(per_class_tp.keys()) + list(per_class_fp.keys()) + list(per_class_fn.keys()))

    print(f"{'Class':<30} {'Precision':>10} {'Recall':>10} {'F1':>10} {'TP':>5} {'FP':>5} {'FN':>5}")
    print("-" * 80)

    total_tp = total_fp = total_fn = 0
    for cls_id in sorted(all_classes):
        tp = per_class_tp[cls_id]
        fp = per_class_fp[cls_id]
        fn = per_class_fn[cls_id]
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        name = CLASS_NAMES.get(cls_id, f"Class {cls_id}")
        print(f"{name:<30} {prec:>10.3f} {rec:>10.3f} {f1:>10.3f} {tp:>5} {fp:>5} {fn:>5}")
        results[cls_id] = {"precision": prec, "recall": rec, "f1": f1}
        total_tp += tp; total_fp += fp; total_fn += fn

    overall_prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    overall_rec  = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    overall_f1   = 2 * overall_prec * overall_rec / (overall_prec + overall_rec) if (overall_prec + overall_rec) > 0 else 0
    print("-" * 80)
    print(f"{'OVERALL':<30} {overall_prec:>10.3f} {overall_rec:>10.3f} {overall_f1:>10.3f}")
    return results


# Load best model weights for evaluation
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("\n=== Test Set Evaluation (IoU=0.5, confidence=0.5) ===\n")
test_results = evaluate_test(model, test_loader)

## 10. Save & Download Model

Saves `dental_model_v2.pth` and triggers automatic download.

**Next step**: Upload this file to the `models/` folder on Replit.

In [ ]:
OUTPUT_PATH = os.path.join(WORK_DIR, "dental_model_v2.pth")

checkpoint = {
    "model_state_dict": best_model_state if best_model_state else model.state_dict(),
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "image_size": IMAGE_SIZE,
    "best_val_loss": best_val_loss,
    "train_losses": train_losses,
    "val_losses": val_losses,
    "test_results": test_results if "test_results" in dir() else {},
}

torch.save(checkpoint, OUTPUT_PATH)
size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"Saved: {OUTPUT_PATH} ({size_mb:.1f} MB)")
print("\nDownloading dental_model_v2.pth...")
files.download(OUTPUT_PATH)
print("\n✅ Done! Upload dental_model_v2.pth to models/ on Replit and restart the server.")